# <font color="#418FDE" size="6.5" uppercase>**Fair vergleichen**</font>

>Last update: 20260825.
    
By the end of this Lecture, you will be able to:
- Vergleichen Modelle fair mit identischen Splits, Baselines und Metriken. 
- Speichern, laden und prüfen Modelle aus verschiedenen Frameworks reproduzierbar. 
- Untersuchen Teilgruppenmetriken, Datenverschiebung und Modellkarten für verantwortungsvollen Einsatz. 


## **1. Fairer Modellvergleich**

### **1.1. Identischer Split**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_20/Lecture_A/image_01_01.jpg?v=1787668614" width="250">



>* Gleiche Daten machen Modellvergleiche fair
>* Metriken bleiben dadurch wirklich vergleichbar

>* Gleiche Beobachtungen für alle Modellvergleiche
>* Validierung dokumentieren und Klassen ausgewogen halten

>* Datenleckage durch passende Splits vermeiden
>* Modelle unter gleichen Bedingungen bewerten



In [ ]:
#@title Python-Code - Identischer Split

# Wir vergleichen Modelle mit demselben Datensplit.
# Identische Testdaten machen Ergebnisse fair vergleichbar.
# Die Ausgabe zeigt faire und unfaire Genauigkeiten.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_breast_cancer

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Wir laden einen kleinen Klassifikationsdatensatz.
data = load_breast_cancer()
X = data.data
y = data.target

# Diese Prüfung schützt vor unerwarteten Datenformen.
if X.shape[0] != y.shape[0]:
    raise ValueError("Merkmale und Zielwerte passen nicht zusammen.")

# Ein einziger Split wird für alle Modelle wiederverwendet.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# Beide Kandidaten sehen exakt dieselben Trainingsdaten.
model_a = make_pipeline(
    StandardScaler(), LogisticRegression(C=0.2, max_iter=1000, random_state=42)
)
model_b = make_pipeline(
    StandardScaler(), LogisticRegression(C=2.0, max_iter=1000, random_state=42)
)

# Jetzt ist nur die Modell-Einstellung unterschiedlich.
model_a.fit(X_train, y_train)
model_b.fit(X_train, y_train)

# Die faire Bewertung nutzt dasselbe Testset.
fair_a = accuracy_score(y_test, model_a.predict(X_test))
fair_b = accuracy_score(y_test, model_b.predict(X_test))

# Ein anderer Split kann den Vergleich verzerren.
X_train_2, X_test_2, y_train_2, y_test_2 = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=7
)

# Dieses Ergebnis ist nicht direkt fair vergleichbar.
model_b_unfair = make_pipeline(
    StandardScaler(), LogisticRegression(C=2.0, max_iter=1000, random_state=42)
)
model_b_unfair.fit(X_train_2, y_train_2)

unfair_b = accuracy_score(y_test_2, model_b_unfair.predict(X_test_2))

print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Fair: Modell A auf Testset 42 = {fair_a:.3f}")
print(f"Fair: Modell B auf Testset 42 = {fair_b:.3f}")
print(f"Unfair: Modell B auf Testset 7 = {unfair_b:.3f}")

# Das Diagramm stellt faire und unfaire Werte nebeneinander.
labels = ["A fair", "B fair", "B anderer Split"]
scores = [fair_a, fair_b, unfair_b]
colors = ["#4C78A8", "#4C78A8", "#F58518"]

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(labels, scores, color=colors)
ax.set_title("Identischer Split macht Modellvergleiche fairer")
ax.set_xlabel("Bewertungssituation")
ax.set_ylabel("Genauigkeit")
ax.set_ylim(0.85, 1.0)
plt.show()



### **1.2. Frameworks fair vergleichen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_20/Lecture_A/image_01_02.jpg?v=1787668618" width="250">



>* Vergleiche Modellleistung unter gleichen Bedingungen
>* Nutze gleiche Daten, Merkmale und Metriken

>* Vorverarbeitung passend und transparent gestalten
>* Hyperparameter fair suchen, Zufall kontrollieren

>* Metriken, Datensatz und Schwellenwerte klar festlegen
>* Technische Unterschiede kontrolliert dokumentieren und bewerten



In [ ]:
#@title Python-Code - Frameworks fair vergleichen

# Wir vergleichen Modelle unter identischen Bedingungen.
# Gleiche Splits und Metriken machen Ergebnisse fair.
# Die Grafik zeigt faire und unfaire Genauigkeit.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_breast_cancer

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Wir laden einen kleinen eingebauten Klassifikationsdatensatz.
data = load_breast_cancer()
X = data.data
y = data.target

# Diese Prüfung macht die Beispielannahmen sichtbar.
if X.shape[0] != y.shape[0]:
    raise ValueError("Merkmale und Zielwerte passen nicht zusammen.")

# Ein gemeinsamer Split ist die Grundlage des fairen Vergleichs.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# Die Baseline nutzt denselben Split wie das echte Modell.
baseline = DummyClassifier(strategy="most_frequent", random_state=42)
baseline.fit(X_train, y_train)

# Die Pipeline skaliert nur mit Trainingsdaten und vermeidet Leakage.
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42)
)

model.fit(X_train, y_train)

# Beide Modelle werden mit derselben Testmetrik bewertet.
baseline_accuracy = accuracy_score(y_test, baseline.predict(X_test))
model_accuracy = accuracy_score(y_test, model.predict(X_test))

# Ein unfairer Split kann ein Modell künstlich besser aussehen lassen.
X_train_bad, X_test_bad, y_train_bad, y_test_bad = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=7
)

model_bad = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42)
)

model_bad.fit(X_train_bad, y_train_bad)
unfair_accuracy = accuracy_score(y_test_bad, model_bad.predict(X_test_bad))

print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Baseline, gleicher Testsplit: {baseline_accuracy:.3f}")
print(f"Logistische Regression, gleicher Testsplit: {model_accuracy:.3f}")
print(f"Logistische Regression, anderer Split: {unfair_accuracy:.3f}")

# Die Balken zeigen, warum gleiche Testdaten wichtig sind.
labels = ["Baseline\ngleich", "Modell\ngleich", "Modell\nanderer Split"]
scores = [baseline_accuracy, model_accuracy, unfair_accuracy]

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(labels, scores, color=["gray", "steelblue", "orange"])
ax.set_title("Fairer Vergleich braucht denselben Testsplit")
ax.set_xlabel("Vergleichsfall")
ax.set_ylabel("Genauigkeit")
ax.set_ylim(0, 1)
plt.show()



### **1.3. Zeit und Speicher**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_20/Lecture_A/image_01_03.jpg?v=1787668616" width="250">



>* Genauigkeit allein reicht nicht für Modellvergleiche
>* Laufzeit und Speicher früh mitbewerten

>* Gleiche Bedingungen und Messpunkte festlegen
>* Mehrfach messen und Schwankungen beachten

>* Speicherarten beeinflussen Training, Inferenz und Verteilung
>* Ressourcen und Einsatzkontext fair mitbewerten



In [ ]:
#@title Python-Code - Zeit und Speicher

# Wir vergleichen Modelle mit Zeit und Speicher.
# Gleiche Daten machen den Vergleich fairer.
# Die Tabelle zeigt Genauigkeit und Ressourcen.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Ein fester Split verhindert unfairen Datenvorteil.
data = load_breast_cancer()
X = data.data
Y = data.target

# Diese Prüfung macht die Datengröße bewusst.
if X.shape[0] != len(Y):
    raise ValueError("Merkmale und Zielwerte passen nicht zusammen.")

# Beide Modelle erhalten exakt dieselben Trainingsdaten.
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.3, stratify=Y, random_state=42
)

# Eine einfache Baseline setzt die Messwerte in Kontext.
models = {
    "Baseline": DummyClassifier(strategy="most_frequent"),
    "Logistische Regression": make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=300, random_state=42)
    ),
}

# Wir messen wiederholt, damit Einzelmessungen weniger dominieren.
results = []
for model_name, model in models.items():
    fit_times = []
    predict_times = []
    for repeat in range(5):
        start_fit = pd.Timestamp.now()
        model.fit(X_train, Y_train)
        fit_seconds = (pd.Timestamp.now() - start_fit).total_seconds()
        start_predict = pd.Timestamp.now()
        predictions = model.predict(X_test)
        predict_seconds = (pd.Timestamp.now() - start_predict).total_seconds()
        fit_times.append(fit_seconds)
        predict_times.append(predict_seconds)
    accuracy = accuracy_score(Y_test, predictions)
    memory_kb = X_train.nbytes / 1024
    results.append([model_name, accuracy, np.mean(fit_times), np.mean(predict_times), memory_kb])

# Eine kleine Tabelle verbindet Qualität mit Ressourcen.
columns = ["Modell", "Genauigkeit", "Training s", "Vorhersage s", "Daten KB"]
summary = pd.DataFrame(results, columns=columns)
summary["Genauigkeit"] = summary["Genauigkeit"].round(3)
summary["Training s"] = summary["Training s"].round(5)
summary["Vorhersage s"] = summary["Vorhersage s"].round(5)
summary["Daten KB"] = summary["Daten KB"].round(1)

# Die Version hilft, Ergebnisse später einzuordnen.
print(f"scikit-learn Version: {sklearn.__version__}")
print("Fairer Vergleich: gleicher Split, gleiche Metrik, gleiche Datenmenge.")
print(summary.to_string(index=False))

# Das Diagramm zeigt den Zielkonflikt anschaulich.
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(summary["Modell"], summary["Training s"], color=["gray", "steelblue"])
ax.set_title("Trainingszeit bei gleichem Datensplit")
ax.set_xlabel("Modell")
ax.set_ylabel("Durchschnittliche Trainingszeit in Sekunden")
plt.tight_layout()
plt.show()



## **2. Reproduzierbare Modelle**

### **2.1. Komplexität fair vergleichen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_20/Lecture_A/image_02_01.jpg?v=1787668626" width="250">



>* Leistung immer mit Aufwand und Ressourcen vergleichen
>* Reproduzierbarkeit beim Laden und Prüfen beachten

>* Gleiche Bedingungen für faire Modellvergleiche
>* Praxisnutzen gegen Infrastrukturkosten abwägen

>* Wartbarkeit und Abhängigkeiten beeinflussen Reproduzierbarkeit.
>* Fair bewerten: Leistung, Ressourcen und Prüfbarkeit.



In [ ]:
#@title Python-Code - Komplexität fair vergleichen

# Wir vergleichen Modellkomplexität unter gleichen Bedingungen.
# Reproduzierbarkeit braucht gespeicherte und prüfbare Kennzahlen.
# Die Grafik zeigt Leistung gegen Modellgröße.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# Wir laden einen kleinen, eingebauten Klassifikationsdatensatz.
data = load_breast_cancer()
X = data.data
y = data.target

# Diese Prüfung macht die Datenannahme sichtbar.
if X.shape[0] != y.shape[0]:
    raise ValueError("Merkmale und Zielwerte passen nicht zusammen.")

# Beide Modelle erhalten exakt denselben Split.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

# Die Baseline lernt nur die häufigste Klasse.
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

# Das lineare Modell nutzt dieselben Daten und Skalierung.
linear_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42)
)
linear_model.fit(X_train, y_train)

# Wir messen Leistung und eine einfache Komplexitätsgröße.
baseline_accuracy = accuracy_score(y_test, baseline.predict(X_test))
linear_accuracy = accuracy_score(y_test, linear_model.predict(X_test))

# Die Baseline speichert nur Klassenhäufigkeiten.
baseline_size = baseline.class_prior_.size
linear_step = linear_model.named_steps["logisticregression"]
linear_size = linear_step.coef_.size + linear_step.intercept_.size

# Diese Tabelle wäre Teil einer Modellkarte.
model_names = ["Baseline", "Logistische Regression"]
accuracies = [baseline_accuracy, linear_accuracy]
sizes = [baseline_size, linear_size]

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Trainingsbeispiele: {len(y_train)}, Testbeispiele: {len(y_test)}")
print(f"Baseline: Genauigkeit {baseline_accuracy:.3f}, gespeicherte Zahlen {baseline_size}")
print(f"Logistische Regression: Genauigkeit {linear_accuracy:.3f}, gespeicherte Zahlen {linear_size}")
print("Fairer Vergleich: gleicher Split, gleiche Metrik, sichtbare Komplexität.")

# Die Grafik macht den Zielkonflikt sichtbar.
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(sizes, accuracies, s=120)

# Beschriftungen helfen beim Vergleich der Punkte.
for name, size, accuracy in zip(model_names, sizes, accuracies):
    ax.annotate(name, (size, accuracy), textcoords="offset points", xytext=(6, 6))

ax.set_title("Leistung und Komplexität fair vergleichen")
ax.set_xlabel("Gespeicherte Modellzahlen")
ax.set_ylabel("Testgenauigkeit")
ax.set_ylim(0.5, 1.02)
plt.show()



### **2.2. Versionen speichern**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_20/Lecture_A/image_02_02.jpg?v=1787668628" width="250">



>* Modellversionen brauchen vollständigen Entstehungskontext
>* Fehlende Details gefährden reproduzierbare Vorhersagen

>* Modell-, Daten- und Experimentversionen klar trennen
>* Änderungen eindeutig kennzeichnen und nachvollziehbar vergleichen

>* Versionierung ermöglicht Rückkehr zu geprüften Modellen
>* Dokumentierte Pakete stützen Audits und Freigaben



In [ ]:
#@title Python-Code - Versionen speichern

# Wir speichern eine Modellversion reproduzierbar.
# Metadaten erklären den Entstehungskontext.
# Geladene Vorhersagen werden danach geprüft.

import numpy as np
import pandas as pd
import sklearn
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Wir nutzen einen kleinen, eingebauten Klassifikationsdatensatz.
iris = load_iris()
X = iris.data
y = iris.target

# Diese Prüfung macht die Datenannahme sichtbar.
if X.shape[0] != y.shape[0]:
    raise ValueError("Merkmale und Zielwerte passen nicht zusammen.")

# Ein fester Split gehört zur reproduzierbaren Version.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

# Die Pipeline speichert Vorverarbeitung und Modell gemeinsam.
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=200, random_state=42)
)

# Genau ein Training erzeugt die Modellversion.
model.fit(X_train, y_train)
predictions_before = model.predict(X_test)
accuracy_before = accuracy_score(y_test, predictions_before)

# Dieses Paket simuliert ein versioniertes Modellartefakt im Speicher.
model_version = {
    "model": model,
    "metadata": {
        "model_name": "iris_logistic_regression",
        "data_version": "sklearn_iris_builtin",
        "split_seed": 42,
        "test_size": 0.25,
        "sklearn_version": sklearn.__version__,
    },
}

# Laden bedeutet hier, dass Modell und Metadaten zusammenkommen.
loaded_model = model_version["model"]
loaded_metadata = model_version["metadata"]

# Die Prüfung vergleicht Vorhersagen vor und nach dem Laden.
predictions_after = loaded_model.predict(X_test)
same_predictions = np.array_equal(predictions_before, predictions_after)
accuracy_after = accuracy_score(y_test, predictions_after)

# Eine kleine Tabelle zeigt die wichtigsten Versionsinformationen.
summary = pd.DataFrame(
    [loaded_metadata]
)[["model_name", "data_version", "split_seed", "sklearn_version"]]

print("scikit-learn-Version: " + sklearn.__version__)
print("Gespeicherte Versionsdaten:")
print(summary.to_string(index=False))
print("Genau gleiche Vorhersagen nach dem Laden: " + str(same_predictions))
print("Testgenauigkeit vorher: " + str(round(accuracy_before, 3)))
print("Testgenauigkeit nachher: " + str(round(accuracy_after, 3)))



### **2.3. Modelle sicher laden**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_20/Lecture_A/image_02_03.jpg?v=1787668630" width="250">



>* Herkunft, Version und Integrität prüfen
>* Modelle nur aus freigegebenen Registern laden

>* Geladene Modelle immer fachlich validieren
>* Prüfbeispiele decken stille Fehler auf

>* Klare Regeln für Modellfreigabe und Versionen
>* Laden als geprüften Übergang verstehen



In [ ]:
#@title Python-Code - Modelle sicher laden

# Dieses Beispiel lädt ein Modell kontrolliert.
# Prüfsummen schützen vor falschen Artefakten.
# Prüfbeispiele bestätigen reproduzierbare Vorhersagen.

import numpy as np
import sklearn
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Wir nutzen einen kleinen eingebauten Datensatz.
iris = load_iris()
X = iris.data
y = iris.target

# Eine einfache Prüfung verhindert unerwartete Eingabeformen.
if X.shape[1] != 4:
    raise ValueError("Unerwartete Anzahl von Merkmalen.")

# Der Split ist fest, damit Ergebnisse vergleichbar bleiben.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

# Die Pipeline speichert Vorverarbeitung und Modell gemeinsam.
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=200, random_state=42)
)

# Wir trainieren genau ein kleines Modell.
model.fit(X_train, y_train)

# Ein Artefakt enthält Modell, Metadaten und Prüfwerte.
check_rows = X_test[:5]
expected_predictions = model.predict(check_rows)
expected_accuracy = accuracy_score(y_test, model.predict(X_test))

# Diese einfache Prüfsumme steht für einen Integritätsnachweis.
checksum = int(np.round(np.sum(expected_predictions) * 1000))
artifact = {
    "model": model,
    "sklearn_version": sklearn.__version__,
    "feature_count": X.shape[1],
    "expected_predictions": expected_predictions.copy(),
    "expected_accuracy": expected_accuracy,
    "checksum": checksum,
}

# Sicheres Laden prüft Metadaten vor der Nutzung.
loaded_artifact = artifact
loaded_model = loaded_artifact["model"]

# Die Merkmalsanzahl muss zur Dokumentation passen.
if loaded_artifact["feature_count"] != X.shape[1]:
    raise ValueError("Merkmalsanzahl passt nicht zum Artefakt.")

# Die Prüfsumme muss unverändert sein.
loaded_checksum = int(np.round(np.sum(loaded_artifact["expected_predictions"]) * 1000))
if loaded_checksum != loaded_artifact["checksum"]:
    raise ValueError("Prüfsumme passt nicht zum Artefakt.")

# Feste Prüfbeispiele zeigen reproduzierbares Verhalten.
loaded_predictions = loaded_model.predict(check_rows)
checks_match = np.array_equal(
    loaded_predictions,
    loaded_artifact["expected_predictions"]
)

# Die Testgenauigkeit wird erneut berechnet.
loaded_accuracy = accuracy_score(y_test, loaded_model.predict(X_test))
accuracy_difference = abs(loaded_accuracy - loaded_artifact["expected_accuracy"])

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Metadatenprüfung bestanden: {loaded_artifact['feature_count'] == 4}")
print(f"Prüfsumme bestanden: {loaded_checksum == loaded_artifact['checksum']}")
print(f"Prüfbeispiele identisch: {checks_match}")
print(f"Genauigkeit nach Laden: {loaded_accuracy:.3f}")
print(f"Abweichung zur gespeicherten Genauigkeit: {accuracy_difference:.3f}")



## **3. Verantwortungsvoll prüfen**

### **3.1. Inferenz sicher ausführen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_20/Lecture_A/image_03_01.jpg?v=1787668620" width="250">



>* Vorhersagen kontrolliert und nachvollziehbar ausführen
>* Eingaben prüfen, Fehler sichtbar behandeln

>* Unsicherheit erkennen und menschlich prüfen lassen
>* Schwellenwerte nach Fehlfolgen begründen

>* Vorhersagen nachvollziehbar und datensparsam dokumentieren
>* Versionen testen, überwachen und Veränderungen erkennen



In [ ]:
#@title Python-Code - Inferenz sicher ausführen

# Dieses Beispiel zeigt sichere Inferenz mit Prüfregeln.
# Unsichere Eingaben werden vorhersagbar zurückgewiesen.
# Das Ergebnis zeigt akzeptierte und blockierte Fälle.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Wir nutzen einen kleinen, eingebauten Klassifikationsdatensatz.
data = load_breast_cancer(as_frame=True)
features = data.frame[data.feature_names]
target = data.target

# Der Split bleibt reproduzierbar und fair prüfbar.
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.25, stratify=target, random_state=42
)

# Vorverarbeitung und Modell bleiben in einer Pipeline verbunden.
model = make_pipeline(
    StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)
)
model.fit(X_train, y_train)

# Trainingsbereiche dienen als einfache Eingabeprüfung.
train_min = X_train.min()
train_max = X_train.max()
required_columns = list(X_train.columns)

# Diese Funktion begrenzt Inferenz auf plausible Eingaben.
def safe_predict_one(row, threshold):
    if list(row.index) != required_columns:
        return "blockiert", np.nan
    if row.isna().any():
        return "blockiert", np.nan
    if ((row < train_min) | (row > train_max)).any():
        return "blockiert", np.nan
    probability = model.predict_proba(pd.DataFrame([row]))[0, 1]
    if probability < threshold and probability > 1 - threshold:
        return "menschlich prüfen", probability
    return "akzeptiert", probability

# Drei Testfälle zeigen typische Inferenzentscheidungen.
valid_row = X_test.iloc[0].copy()
missing_row = valid_row.copy()
missing_row.iloc[0] = np.nan
shifted_row = valid_row.copy()
shifted_row.iloc[1] = train_max.iloc[1] * 3

# Die Ergebnisse bleiben kurz und nachvollziehbar.
examples = {
    "gültig": valid_row,
    "fehlender Wert": missing_row,
    "außerhalb Bereich": shifted_row,
}

# Wir bewerten zusätzlich die normale Testgenauigkeit.
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Testgenauigkeit: {accuracy:.3f}")

# Jede Inferenz erhält einen Status statt blinder Vorhersage.
statuses = []
for name, row in examples.items():
    status, probability = safe_predict_one(row, threshold=0.60)
    statuses.append(status)
    shown_probability = "nicht berechnet" if np.isnan(probability) else f"{probability:.3f}"
    print(f"{name}: {status}, Wahrscheinlichkeit={shown_probability}")

# Das Balkendiagramm macht blockierte Fälle sichtbar.
status_names = ["akzeptiert", "menschlich prüfen", "blockiert"]
status_counts = [statuses.count(name) for name in status_names]
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(status_names, status_counts, color=["#4c78a8", "#f58518", "#e45756"])
ax.set_title("Sichere Inferenz: Status je Eingabe")
ax.set_xlabel("Inferenzstatus")
ax.set_ylabel("Anzahl Beispiele")
plt.show()



### **3.2. Teilgruppen fair bewerten**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_20/Lecture_A/image_03_02.jpg?v=1787668622" width="250">



>* Nicht nur Durchschnittswerte betrachten
>* Leistung für relevante Gruppen getrennt prüfen

>* Relevante Gruppen sorgfältig und kontextbezogen wählen
>* Fehlerarten und Gruppengrößen gemeinsam bewerten

>* Teilgruppen regelmäßig prüfen und vergleichen
>* Verschiebungen transparent in Modellkarten dokumentieren



In [ ]:
#@title Python-Code - Teilgruppen fair bewerten

# Wir prüfen Modellleistung getrennt nach Teilgruppen.
# Gleiche Testdaten zeigen versteckte Qualitätsunterschiede.
# Die Grafik macht faire Bewertung sichtbar.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# Diese Daten enthalten eine künstliche, bekannte Teilgruppe.
rng = np.random.default_rng(42)
features, target = make_classification(
    n_samples=900,
    n_features=6,
    n_informative=4,
    n_redundant=0,
    class_sep=1.0,
    random_state=42,
)

# Die Teilgruppe steht für eine relevante Nutzungssituation.
group = np.where(features[:, 0] > 0.2, "Gruppe A", "Gruppe B")
noisy_target = target.copy()
mask = (group == "Gruppe B") & (rng.random(len(target)) < 0.22)
noisy_target[mask] = 1 - noisy_target[mask]

# Wir teilen Daten fair und stratifiziert in Training und Test.
(
    x_train,
    x_test,
    y_train,
    y_test,
    group_train,
    group_test,
) = train_test_split(
    features,
    noisy_target,
    group,
    test_size=0.30,
    stratify=noisy_target,
    random_state=42,
)

# Eine einfache Regression reicht für den Vergleich.
model = LogisticRegression(max_iter=500, random_state=42)
model.fit(x_train, y_train)
prediction = model.predict(x_test)

# Wir prüfen zuerst die Gesamtgenauigkeit.
overall_accuracy = accuracy_score(y_test, prediction)
print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Gesamtgenauigkeit: {overall_accuracy:.3f}")

# Danach berechnen wir dieselbe Metrik pro Teilgruppe.
rows = []
for group_name in sorted(np.unique(group_test)):
    group_mask = group_test == group_name
    group_accuracy = accuracy_score(y_test[group_mask], prediction[group_mask])
    rows.append({"Teilgruppe": group_name, "Genauigkeit": group_accuracy})

# Gruppengrößen helfen, Kennzahlen vorsichtig zu interpretieren.
for row in rows:
    group_mask = group_test == row["Teilgruppe"]
    row["Anzahl"] = int(np.sum(group_mask))

result_table = pd.DataFrame(rows)
print("Teilgruppen: " + ", ".join(result_table["Teilgruppe"].tolist()))
print("Kleinere Gruppen brauchen besonders vorsichtige Interpretation.")

# Die Balken zeigen, ob der Durchschnitt Unterschiede verdeckt.
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(result_table["Teilgruppe"], result_table["Genauigkeit"], color="#4C78A8")
ax.axhline(overall_accuracy, color="#F58518", linestyle="--", label="Gesamt")

# Beschriftungen machen die Gruppengröße direkt sichtbar.
for index, row in result_table.iterrows():
    label = f"n={row['Anzahl']}"
    ax.text(index, row["Genauigkeit"] + 0.02, label, ha="center")

ax.set_ylim(0, 1)
ax.set_title("Genauigkeit nach Teilgruppe")
ax.set_xlabel("Teilgruppe")
ax.set_ylabel("Genauigkeit")
ax.legend()
plt.show()



### **3.3. Modellkarte erstellen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_20/Lecture_A/image_03_03.jpg?v=1787668624" width="250">



>* Modellkarten erklären Zweck, Prüfung und Einsatzbedingungen
>* Klare Grenzen verhindern falsche Nutzung

>* Evaluation mit Splits, Metriken und Teilgruppen zeigen
>* Datenverschiebungen und Überwachungsbedarf klar benennen

>* Betriebshinweise und Nachprüfungen klar festhalten
>* Grenzen, Verantwortung und Vertrauen nachvollziehbar machen



In [ ]:
#@title Python-Code - Modellkarte erstellen

# Diese Übung erstellt eine einfache Modellkarte.
# Teilgruppenmetriken zeigen mögliche Fairnessrisiken sichtbar.
# Am Ende entsteht eine kompakte Einsatzempfehlung.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# Wir erzeugen kleine, reproduzierbare Beispieldaten.
features, target = make_classification(
    n_samples=600,
    n_features=6,
    n_informative=4,
    random_state=42,
)

# Eine Teilgruppe simuliert unterschiedliche Einsatzbedingungen.
rng = np.random.default_rng(42)
group = np.where(features[:, 0] > np.median(features[:, 0]), "Gruppe A", "Gruppe B")

# Wir prüfen eine einfache Annahme zur Datenform.
if features.shape[0] != target.shape[0]:
    raise ValueError("Merkmale und Zielwerte passen nicht zusammen.")

# Der Split bleibt für alle Modellkartenwerte identisch.
X_train, X_test, y_train, y_test, group_train, group_test = train_test_split(
    features, target, group, test_size=0.3, stratify=target, random_state=42
)

# Ein kleines Modell reicht für die Dokumentationsidee.
model = LogisticRegression(max_iter=300, random_state=42)
model.fit(X_train, y_train)

# Wir berechnen Gesamtleistung und Teilgruppenleistung.
predictions = model.predict(X_test)
overall_accuracy = accuracy_score(y_test, predictions)
baseline_accuracy = max(np.mean(y_test == 0), np.mean(y_test == 1))

# Die Modellkarte sammelt technische und fachliche Hinweise.
rows = []
for group_name in sorted(np.unique(group_test)):
    mask = group_test == group_name
    rows.append([group_name, int(mask.sum()), accuracy_score(y_test[mask], predictions[mask])])

# Eine kleine Tabelle macht Unterschiede schnell sichtbar.
subgroup_table = pd.DataFrame(rows, columns=["Teilgruppe", "Fälle", "Accuracy"])
subgroup_table["Accuracy"] = subgroup_table["Accuracy"].round(3)

# Wir leiten eine einfache Warnung aus der größten Lücke ab.
accuracy_gap = subgroup_table["Accuracy"].max() - subgroup_table["Accuracy"].min()
risk_note = "Nachprüfung nötig" if accuracy_gap > 0.08 else "Keine große Lücke"

print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Zweck: Demo-Modellkarte für eine binäre Klassifikation.")
print(f"Baseline Accuracy: {baseline_accuracy:.3f}")
print(f"Modell Accuracy: {overall_accuracy:.3f}")
print(f"Teilgruppenlücke: {accuracy_gap:.3f} ({risk_note})")
print("Modellkarte: Nutzung nur mit Monitoring und erneuter Prüfung bei Datendrift.")

# Das Diagramm zeigt den Modellkartenabschnitt zu Teilgruppen.
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(subgroup_table["Teilgruppe"], subgroup_table["Accuracy"], color=["#4C78A8", "#F58518"])
ax.axhline(overall_accuracy, color="black", linestyle="--", label="Gesamtwert")

ax.set_title("Modellkarte: Accuracy nach Teilgruppe")
ax.set_xlabel("Teilgruppe")
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1)
ax.legend()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Fair vergleichen**</font>


In this lecture, you learned to:
- Vergleichen Modelle fair mit identischen Splits, Baselines und Metriken. 
- Speichern, laden und prüfen Modelle aus verschiedenen Frameworks reproduzierbar. 
- Untersuchen Teilgruppenmetriken, Datenverschiebung und Modellkarten für verantwortungsvollen Einsatz. 

<font color='yellow'>Congratulations on completing this course!</font>